# 1) M2.1 Market clearing — Exercises

**Goals:** aggregate curves, a complete uniform-price clearing algorithm (including the case where a whole price range clears), the welfare LP and its dual, consumer and producer surplus, and a first look at the merit-order effect of extra wind — checked with PyPSA.

Prices in €/MWh, quantities in MW, welfare and surplus in €/h. The setup cell holds the lecture's synthetic order book.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import numpy as np
import pandas as pd
import pypsa
from scipy.optimize import linprog

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

OFFERS = pd.DataFrame({
    "order": ["solar", "wind", "run_of_river", "nuclear_share", "coal", "ccgt", "imports", "ocgt"],
    "price": [-5.0, 0.0, 5.0, 12.0, 95.0, 110.0, 120.0, 165.0],
    "qty":   [300.0, 400.0, 500.0, 350.0, 210.0, 500.0, 600.0, 250.0],
})
BIDS = pd.DataFrame({
    "order": ["inflexible_load", "industry_A", "industry_B", "pumped_storage", "electrolyser"],
    "price": [3000.0, 140.0, 115.0, 60.0, 40.0],
    "qty":   [1700.0, 200.0, 150.0, 120.0, 100.0],
})

# a tiny order book whose curves cross on a vertical step
TINY_OFFERS = pd.DataFrame({"order": ["A", "B"], "price": [10.0, 30.0], "qty": [100.0, 100.0]})
TINY_BIDS = pd.DataFrame({"order": ["X", "Y"], "price": [50.0, 20.0], "qty": [100.0, 100.0]})

### Warm-ups

**1. Aggregate curves**

Write `aggregate_curve(orders, side)` returning a DataFrame sorted in merit order (offers ascending by price, bids descending) with a `cum_qty` column (MW) giving the cumulative quantity at the **end** of each step.

In [2]:
def aggregate_curve(orders: pd.DataFrame, side: str) -> pd.DataFrame:
    """Orders in merit order with cumulative quantity; side is 'supply' or 'demand'."""
    ### BEGIN SOLUTION
    out = orders.sort_values("price", ascending=(side == "supply"), kind="stable").reset_index(drop=True)
    out["cum_qty"] = out.qty.cumsum()
    return out
    ### END SOLUTION

In [3]:
supply = aggregate_curve(OFFERS, "supply")
demand = aggregate_curve(BIDS, "demand")
assert supply.price.is_monotonic_increasing and demand.price.is_monotonic_decreasing
assert supply.cum_qty.iloc[-1] == OFFERS.qty.sum() and demand.cum_qty.iloc[2] == 2050

**2. A complete clearing algorithm**

Write `clear(offers, bids)` returning a dict with `volume`, `price`, and `price_range` (low, high). Walk the curves as in the lecture. The price rule:
- a partly accepted order sets the price (and the range collapses to that price);
- otherwise every price in a range clears the same volume — the range is bounded by the last accepted and first rejected orders on each side — and you return its **midpoint**;
- if nothing trades, return volume 0 and price `None`.

In [4]:
def clear(offers: pd.DataFrame, bids: pd.DataFrame) -> dict:
    """Uniform-price clearing with a midpoint rule for price ranges."""
    ### BEGIN SOLUTION
    o = aggregate_curve(offers, "supply")
    b = aggregate_curve(bids, "demand")
    left_o, left_b = o.qty.to_numpy().copy(), b.qty.to_numpy().copy()
    i = j = 0
    while i < len(o) and j < len(b) and o.price[i] <= b.price[j]:
        q = min(left_o[i], left_b[j])
        left_o[i] -= q
        left_b[j] -= q
        if left_o[i] == 0:
            i += 1
        if left_b[j] == 0:
            j += 1
    sold, bought = o.qty.to_numpy() - left_o, b.qty.to_numpy() - left_b
    volume = sold.sum()
    if volume == 0:
        return {"volume": 0.0, "price": None, "price_range": None}
    partial_o = [k for k in range(len(o)) if 0 < sold[k] < o.qty[k]]
    partial_b = [k for k in range(len(b)) if 0 < bought[k] < b.qty[k]]
    if partial_o:
        p = float(o.price[partial_o[0]])
        return {"volume": volume, "price": p, "price_range": (p, p)}
    if partial_b:
        p = float(b.price[partial_b[0]])
        return {"volume": volume, "price": p, "price_range": (p, p)}
    acc_o, acc_b = np.flatnonzero(sold > 0), np.flatnonzero(bought > 0)
    low = max(o.price[acc_o[-1]], b.price[acc_b[-1] + 1] if acc_b[-1] + 1 < len(b) else -np.inf)
    high = min(b.price[acc_b[-1]], o.price[acc_o[-1] + 1] if acc_o[-1] + 1 < len(o) else np.inf)
    return {"volume": volume, "price": float(low + high) / 2, "price_range": (float(low), float(high))}
    ### END SOLUTION

In [5]:
r = clear(OFFERS, BIDS)
assert np.isclose(r["volume"], 2050) and np.isclose(r["price"], 110), "the lecture's result"
t = clear(TINY_OFFERS, TINY_BIDS)
assert np.isclose(t["volume"], 100) and t["price_range"] == (20.0, 30.0) and np.isclose(t["price"], 25.0)
none = clear(TINY_OFFERS.assign(price=[60.0, 70.0]), TINY_BIDS)
assert none["volume"] == 0 and none["price"] is None, "offers above every bid: no trade"
print(f"lecture book: {r['volume']:.0f} MW at {r['price']:.0f} | tiny book: {t['volume']:.0f} MW, "
      f"any price in {t['price_range']}, midpoint {t['price']:.0f} EUR/MWh")

lecture book: 2050 MW at 110 | tiny book: 100 MW, any price in (20.0, 30.0), midpoint 25 EUR/MWh


### Core

**3. The welfare LP**

Write `welfare_lp(offers, bids)` returning a dict with `volume`, `price` (the dual of the balance constraint), `welfare` (€/h) and the accepted quantities `sold` and `bought` (arrays in the input order). For the tiny book, check that the LP's price lies inside your price range.

In [6]:
def welfare_lp(offers: pd.DataFrame, bids: pd.DataFrame) -> dict:
    """Market clearing as welfare maximisation with HiGHS."""
    ### BEGIN SOLUTION
    n_o = len(offers)
    res = linprog(np.r_[offers.price, -bids.price], A_eq=[np.r_[np.ones(n_o), -np.ones(len(bids))]], b_eq=[0.0],
                  bounds=list(zip(np.zeros(n_o + len(bids)), np.r_[offers.qty, bids.qty])), method="highs")
    return {"volume": res.x[:n_o].sum(), "price": res.eqlin.marginals[0], "welfare": -res.fun,
            "sold": res.x[:n_o], "bought": res.x[n_o:]}
    ### END SOLUTION

In [7]:
lp = welfare_lp(OFFERS, BIDS)
assert np.isclose(lp["volume"], 2050) and np.isclose(lp["price"], 110) and np.isclose(lp["welfare"], 5_088_200)
lp_tiny = welfare_lp(TINY_OFFERS, TINY_BIDS)
low, high = clear(TINY_OFFERS, TINY_BIDS)["price_range"]
assert np.isclose(lp_tiny["volume"], 100) and low - 1e-9 <= lp_tiny["price"] <= high + 1e-9
print(f"tiny book: the LP reports {lp_tiny['price']:.2f} EUR/MWh, inside the range {low:.0f}-{high:.0f}")

tiny book: the LP reports 20.00 EUR/MWh, inside the range 20-30


**4. Surplus**

Write `surplus(offers, bids, price, sold, bought)` returning (consumer surplus, producer surplus) in €/h. Their sum must equal the LP welfare — for any price inside the clearing range.

In [8]:
def surplus(offers, bids, price, sold, bought) -> tuple[float, float]:
    """Consumer and producer surplus (EUR/h) at a uniform price."""
    ### BEGIN SOLUTION
    consumer = float(np.sum((bids.price.to_numpy() - price) * np.asarray(bought)))
    producer = float(np.sum((price - offers.price.to_numpy()) * np.asarray(sold)))
    return consumer, producer
    ### END SOLUTION

In [9]:
cs, ps = surplus(OFFERS, BIDS, lp["price"], lp["sold"], lp["bought"])
assert np.isclose(cs + ps, lp["welfare"])
for p in [20.0, 25.0, 30.0]:                      # any price in the tiny book's range
    c, s = surplus(TINY_OFFERS, TINY_BIDS, p, lp_tiny["sold"], lp_tiny["bought"])
    assert np.isclose(c + s, lp_tiny["welfare"]), "welfare does not depend on the price - only its split does"
print(f"lecture book: consumer surplus {cs:,.0f} EUR/h, producer surplus {ps:,.0f} EUR/h")

lecture book: consumer surplus 4,919,750 EUR/h, producer surplus 168,450 EUR/h


**5. Interpretation**

In the tiny book every price from 20 to 30 €/MWh clears the same 100 MW, and the welfare is the same at all of them. So what *does* the choice of price within the range decide, who cares about it, and why must an exchange publish a deterministic rule for it rather than let the solver choose?

<!-- BEGIN SOLUTION -->
Within the range the *physical* outcome — who produces, who consumes, total welfare — is fixed; the price only decides how that welfare is **split** between buyers and sellers. At 20 €/MWh buyer X keeps more surplus (it pays 20 for power it valued at 50); at 30 €/MWh seller A keeps more (it receives 30 for power it offered at 10). That split is real money, so every participant cares, and it must not depend on accidents of the solver: different algorithms (or the same solver with different settings) can return different duals inside the range, as the LP did here. A published, deterministic rule (e.g. a midpoint or a rule based on the orders' positions) makes prices reproducible, auditable and the same for everyone, which is essential for trust in the market and for settling financial contracts that reference the price. The European day-ahead algorithm applies such rules for exactly this degenerate case (Maths-2).
<!-- END SOLUTION -->

### Challenge

**6. 800 MW more wind**

Add a second wind offer of 800 MW at 0 €/MWh. Write `compare_with_extra_wind(extra_MW)` returning a DataFrame with the price, volume, consumer surplus and producer surplus **of the original offers only** (excluding the new wind), before and after. Check the new price with PyPSA (demand bids as `sign=-1` generators with `marginal_cost = -bid`).

In [10]:
def compare_with_extra_wind(extra_MW: float) -> pd.DataFrame:
    """Price, volume and surpluses before and after adding a zero-price wind offer."""
    ### BEGIN SOLUTION
    rows = {}
    for label, offers in [("before", OFFERS),
                          ("after", pd.concat([OFFERS, pd.DataFrame({"order": ["new_wind"], "price": [0.0],
                                                                     "qty": [extra_MW]})], ignore_index=True))]:
        r = welfare_lp(offers, BIDS)
        cs, _ = surplus(offers, BIDS, r["price"], r["sold"], r["bought"])
        old = slice(0, len(OFFERS))                          # surplus of the original plants only
        ps_old = float(np.sum((r["price"] - OFFERS.price.to_numpy()) * r["sold"][old]))
        rows[label] = {"price": r["price"], "volume": r["volume"], "consumer_surplus": cs,
                       "producer_surplus_existing": ps_old}
    return pd.DataFrame(rows).T
    ### END SOLUTION


def pypsa_price(offers: pd.DataFrame, bids: pd.DataFrame) -> float:
    n = pypsa.Network()
    n.set_snapshots([0])
    n.add("Bus", "HR")
    for r in offers.itertuples():
        n.add("Generator", r.order, bus="HR", p_nom=r.qty, marginal_cost=r.price)
    for r in bids.itertuples():
        n.add("Generator", r.order, bus="HR", p_nom=r.qty, sign=-1, marginal_cost=-r.price)
    n.optimize(solver_name="highs", log_to_console=False)
    return float(n.buses_t.marginal_price.iloc[0, 0])

In [11]:
table = compare_with_extra_wind(800)
assert np.isclose(table.loc["before", "price"], 110) and np.isclose(table.loc["after", "price"], 12)
assert table.loc["after", "volume"] > table.loc["before", "volume"], "cheaper power serves the flexible bids"
assert table.loc["after", "consumer_surplus"] > table.loc["before", "consumer_surplus"]
assert table.loc["after", "producer_surplus_existing"] < table.loc["before", "producer_surplus_existing"]
with_wind = pd.concat([OFFERS, pd.DataFrame({"order": ["new_wind"], "price": [0.0], "qty": [800.0]})], ignore_index=True)
assert np.isclose(pypsa_price(with_wind, BIDS), table.loc["after", "price"]), "PyPSA agrees"
table.round(0)

        price  volume  consumer_surplus  producer_surplus_existing
before  110.0  2050.0         4919750.0                   168450.0
after    12.0  2270.0         5129210.0                    13400.0

**7. Interpretation**

Adding 800 MW of zero-cost wind drops the price from 110 to 12 €/MWh. Who gains and who loses, by how much, and what does this mean for the revenues of *existing* plants — including the wind farms that were already there? Why is this a problem for investment in more wind (a preview of M2.2's "cannibalisation")?

<!-- BEGIN SOLUTION -->
The extra wind pushes the supply curve to the right, so the intersection moves down the demand curve: the marginal order is no longer the CCGT (110 €/MWh) but nuclear (12 €/MWh) — the **merit-order effect**. Consumers gain twice: everything they already bought is cheaper, and the flexible bids (pumped storage, electrolyser) are now served, raising volume from 2,050 to 2,270 MW. Existing producers lose most of their inframarginal rent: run-of-river, for example, earned 105 €/MWh above its offer and now earns 7; coal and the CCGT are pushed out entirely — the producer surplus of the original plants collapses from about 168,000 to 13,000 €/h. The existing wind and solar farms are hit too: they still produce the same energy, but at a far lower price. That is **cannibalisation**: the more zero-cost renewables are built, the lower the price precisely in the hours they produce, so each additional wind farm earns less per MWh than the previous one. Without flexibility (storage, flexible demand, exports) that absorbs the cheap hours, the market value of new wind and solar falls, weakening the investment case — one reason for support schemes such as contracts for difference, and the topic of M2.2.
<!-- END SOLUTION -->